# 04 — Bayesian Elasticity Estimation

Show the prior structure and (optionally) fit the hierarchical model. **Requires PyMC.**

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from phillyparking.elasticity.priors import central_prior, named_scenario, cross_priors
print('central prior:', central_prior())
print('low scenario :', named_scenario('low'))
print('high scenario:', named_scenario('high'))
print('cross priors :', cross_priors())


## Build a synthetic panel with known elasticities (3 zones × 2 neighborhoods)

In [ ]:
rng = np.random.default_rng(7)
true_elast = {'CCC': -0.45, 'UC': -0.32, 'NL': -0.55}
rows = []
for zone, e_z in true_elast.items():
    for nh in ['A', 'B']:
        for t in range(120):
            log_p = rng.normal(np.log(2.5), 0.25)
            log_q = -1.0 + e_z * log_p + rng.normal(0, 0.05)
            rows.append({'zone': zone, 'nbhd': nh, 't': t,
                         'log_price': log_p, 'log_q': log_q})
panel = pd.DataFrame(rows)
panel.head()


## Fit hierarchical model

**WARNING: requires PyMC, takes ~1–2 minutes.**

In [ ]:
try:
    from phillyparking.elasticity.hierarchical_bayes import fit_elasticity_model
    import arviz as az
    idata = fit_elasticity_model(panel, draws=500, tune=500, chains=2)
    az.plot_posterior(idata)
    plt.show()
except ImportError as e:
    print('Install pymc and arviz to run this section:', e)
    idata = None


## Posterior zone elasticities vs. ground truth

In [ ]:
if idata is not None:
    try:
        from phillyparking.elasticity.hierarchical_bayes import posterior_zone_elasticity
        post = posterior_zone_elasticity(idata)
        print(post)
        print('truth:', true_elast)
    except Exception as e:
        print('posterior summary unavailable:', e)
else:
    print('Skipping — no posterior available.')


## Next steps

- Use the fitted (or central) priors to drive `05_seattle_pricing_simulation.ipynb`.
